# Red-team sweep on a Colab GPU

Runtime → Change runtime type → **A100 GPU**. Ollama serves the models on the GPU inside this VM. The harness runs many attempts in parallel against several target models, then `analysis.py` pools the results into success-rate distributions with 95% confidence intervals.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
curl -fsSL https://ollama.com/install.sh | sh

`OLLAMA_NUM_PARALLEL` is how many requests each loaded model serves at once; keep `WORKERS` below it. Two loaded models = the attacker/judge plus the current target.

In [ ]:
%%bash
export OLLAMA_NUM_PARALLEL=8 OLLAMA_MAX_LOADED_MODELS=2 OLLAMA_KEEP_ALIVE=1h
nohup ollama serve > /content/ollama.log 2>&1 &
sleep 5 && curl -s localhost:11434/api/version

In [ ]:
ATTACKER = 'dolphin-mistral'
TARGETS = ['llama3:8b', 'mistral:7b', 'qwen2.5:7b', 'gemma2:9b']
for m in [ATTACKER, *TARGETS]:
    !ollama pull {m} > /dev/null && echo pulled {m}

In [ ]:
%%bash
cd /content
[ -d multi-agent-red-team ] || git clone https://github.com/guynutman/multi-agent-red-team.git
cd multi-agent-red-team && git pull -q
pip -q install uv
uv sync -q
uv run pytest -q -m 'not ollama'

## Sweep
`TRIALS` attempts per sub-agent per target. 9 sub-agents x 50 trials = 450 attempts per target. Multi-turn agents make up to `--max-turns` target calls each. The same `SEED` is used for every target, so each model faces the same sequence of seeds.

In [ ]:
import subprocess, time
%cd /content/multi-agent-red-team
AGENTS = 'basic_direct,external_indirect,system_prompt_leak,secret_leak,tool_hijack,crescendo_leak,rag_poisoning,sycophancy,unbounded_output'
TRIALS, WORKERS, SEED = 50, 8, 1000
for target in TARGETS:
    t0 = time.time()
    subprocess.run([
        'uv', 'run', 'python', 'main.py',
        '--campaign-name', f'sweep_{target}', '--target', f'ollama:{target}',
        '--attacker', f'ollama:{ATTACKER}', '--sub-agents', AGENTS,
        '--trials-per-agent', str(TRIALS), '--workers', str(WORKERS), '--seed', str(SEED),
        '--max-turns', '4', '--db-path', 'campaigns.db', '--log-level', 'WARNING',
        '--output-report', f'reports/sweep_{target.replace(":", "_")}.md',
    ], check=True)
    print(f'{target}: {time.time() - t0:.0f}s')

In [ ]:
!uv run python analysis.py --db campaigns.db -o reports/analysis.md --csv reports/attempts.csv
from IPython.display import Markdown
Markdown(open('reports/analysis.md').read())

## Plot: success rate by category, per target model (95% Wilson CI)

In [ ]:
import sys; sys.path.insert(0, '.')
import pandas as pd, matplotlib.pyplot as plt
from stats import wilson
from taxonomy import OWASP_LLM_TOP10

df = pd.read_csv('reports/attempts.csv')
scored = df[df.verdict.isin(['success', 'failure', 'unclear'])]
g = scored.groupby(['owasp_category', 'target_model']).success.agg(['sum', 'count']).reset_index()
g[['lo', 'hi']] = g.apply(lambda r: pd.Series(wilson(int(r['sum']), int(r['count']))), axis=1)
g['rate'] = g['sum'] / g['count']

COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']   # fixed categorical order
models = sorted(g.target_model.unique())
cats = sorted(g.owasp_category.unique())
fig, ax = plt.subplots(figsize=(9, 0.6 * len(cats) + 1.5))
step = 0.8 / max(1, len(models))
for i, m in enumerate(models):
    d = g[g.target_model == m].set_index('owasp_category').reindex(cats)
    y = [k + (i - (len(models) - 1) / 2) * step for k in range(len(cats))]
    ax.errorbar(d.rate, y, xerr=[d.rate - d.lo, d.hi - d.rate], fmt='o', ms=7, lw=2,
                capsize=0, color=COLORS[i % len(COLORS)], label=m.replace('ollama:', ''))
ax.set_yticks(range(len(cats)), [f"{c} {OWASP_LLM_TOP10[c].name}" if c in OWASP_LLM_TOP10 else c for c in cats])
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(lambda v, _: f'{v:.0%}')
ax.grid(axis='x', color='#e6e5e0'); ax.set_axisbelow(True)
for s in ('top', 'right', 'left'): ax.spines[s].set_visible(False)
ax.set_xlabel('attack success rate (95% Wilson CI)')
ax.legend(title='target', frameon=False, loc='lower right')
ax.set_title('Attack success by OWASP category and target model', loc='left')
plt.tight_layout(); plt.savefig('reports/success_by_category.png', dpi=160); plt.show()

Distributions beyond rates, for example output tokens for LLM10 and turns used for multi-turn attacks:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for i, m in enumerate(models):
    d = df[(df.target_model == m) & (df.sub_agent_type == 'unbounded_output')].output_tokens.dropna()
    axes[0].hist(d, bins=20, alpha=0.6, color=COLORS[i % len(COLORS)], label=m)
    t = df[(df.target_model == m) & df.turns_used.notna()].turns_used
    axes[1].hist(t + (i - 1.5) * 0.15, bins=16, alpha=0.7, color=COLORS[i % len(COLORS)], label=m)
axes[0].set_title('LLM10 output tokens (cap 1024)', loc='left'); axes[1].set_title('Turns used (multi-turn agents)', loc='left')
for a in axes:
    for s in ('top', 'right'): a.spines[s].set_visible(False)
axes[0].legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
from google.colab import files
!cd reports && zip -q -r /content/results.zip . ../campaigns.db
files.download('/content/results.zip')

### Notes
- **Attacker refusals** show up as `error` rows and are excluded from success rates. Check the error column: a high error rate means the attacker, not the target, is limiting the sample.
- **Bigger targets:** an A100 40GB fits a 4-bit 70B alone (`llama3.1:70b`). Pair it with a hosted attacker/judge, e.g. `--attacker gemini:gemini-flash-latest` with `GEMINI_API_KEY` set.
- **Re-running** with the same `SEED` reproduces each attempt's seeds. Ollama sampling is seeded, so results should match closely but are not guaranteed bit-identical across GPUs.